In [ ]:
import polars as pl

In [33]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the console or notebook output.
    
    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        
        print(df)

In [16]:
off_target_reps = (
    pl.scan_csv([
        "/mnt/hpccs01/work/microbiome/db/gtdb/gtdb_release226/ar53_metadata_r226.tsv.gz",
        "/mnt/hpccs01/work/microbiome/db/gtdb/gtdb_release226/bac120_metadata_r226.tsv.gz"],
        separator='\t', has_header=True, ignore_errors=True
        )
    .select(['accession', 'gtdb_taxonomy', 'checkm2_completeness', 'checkm2_contamination', 'gtdb_representative'])
    .filter(pl.col('gtdb_representative') == 't')
    .with_columns(
        pl.col('gtdb_taxonomy').str.split(';').list.get(-3).alias('family')
    )
    .sort(['family', 'checkm2_completeness', 'checkm2_contamination'], descending=[False, True, False])
    .group_by('family')
    .head(1)
    .collect()
    ['accession']
    .to_list()
    )


print(len(off_target_reps))

5932
